In [ ]:
# ============================================================
# Colab: RL for simplifying "hard" unknots (CSV→GCS first-col only)
# ============================================================

# --------- 0) Install dependencies ----------
!pip -q install "gymnasium>=0.29" "stable-baselines3>=2.3.0" spherogram gcsfs numpy tqdm
!pip -q install snappy-manifolds

# --------- 1) Imports & seeds ----------
import os, re, json, random, math, io, gzip, csv, time, itertools, sys
import numpy as np
from typing import List, Iterable, Any, Optional
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
from tqdm.auto import tqdm
import numpy as np

import snappy
from spherogram import Link
import gcsfs

from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import configure

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# --------- 2) CSV → GCS reader (first column only) ----------
def _normalize_gcs_path(path: str) -> str:
    # gcsfs usually expects "bucket/path". Accept "gs://bucket/path" as well.
    return path[5:] if path.startswith("gs://") else path

def read_first_column_gcs(path: str,
                          max_lines: int | None = None,
                          has_header: bool = True,
                          delimiter: str | None = None,
                          encoding: str = "utf-8") -> list[str]:
    """
    Read a CSV (optionally .gz) from GCS and return only the first column as strings.
    - path: e.g. "gs://bucket/folder/file.csv" or "bucket/folder/file.csv"
    - max_lines: cap rows returned (after header handling)
    - has_header: skip first row if True
    - delimiter: set explicitly ("," or "\t"); if None, sniff a small sample
    - encoding: for non-gz files
    """
    fs = gcsfs.GCSFileSystem(token="anon")
    path = _normalize_gcs_path(path)
    results: list[str] = []

    def make_reader(fh):
        nonlocal delimiter
        if delimiter is None:
            pos = fh.tell()
            sample = fh.read(4096)
            fh.seek(pos)
            try:
                dialect = csv.Sniffer().sniff(sample)
            except Exception:
                dialect = csv.excel  # default (,)
            return csv.reader(fh, dialect)
        else:
            return csv.reader(fh, delimiter=delimiter)

    if path.lower().endswith(".gz"):
        with fs.open(path, "rb") as fb:
            with gzip.GzipFile(fileobj=fb, mode="rb") as gz:
                with io.TextIOWrapper(gz, encoding=encoding, newline="") as ft:
                    reader = make_reader(ft)
                    if has_header:
                        next(reader, None)
                    for i, row in enumerate(reader):
                        if max_lines is not None and i >= max_lines: break
                        if row:
                            results.append(str(row[0]).strip())
    else:
        with fs.open(path, "rt", encoding=encoding, newline="") as f:
            reader = make_reader(f)
            if has_header:
                next(reader, None)
            for i, row in enumerate(reader):
                if max_lines is not None and i >= max_lines: break
                if row:
                    results.append(str(row[0]).strip())
    return results

# --------- 3) Strict PD/DT parser + cleaner ----------
_RE_DT_PREFIX = re.compile(r'^\s*DT\s*:\s*\[', re.I)
_RE_PDLIST = re.compile(r'^\s*\[\s*(\[\s*\d+(?:\s*,\s*\d+){3}\s*\]\s*,?\s*)+\]\s*$') # [[...], ...]
_RE_XPD       = re.compile(r'[Xx]\s*\[')

def parse_link_strict(s: str) -> Link:
    """
    Accept ONLY PD/DT encodings:
      - 'DT: [ ... ]'
      - PD as nested lists: [[8,3,1,4],[2,6,3,5],...]
      - PD in X[...] blocks: X[a,b,c,d], X[...], ...
    Also unwrap simple JSON/JSONL if it contains 'pd' or 'dt' fields.
    """
    t = s.strip()
    if _RE_DT_PREFIX.match(t):
        return Link(t)
    if _RE_PDLIST.match(t):
    # Convert safely into Python object
        try:
            pd_obj = json.loads(t)  # works if it's valid JSON-like
        except json.JSONDecodeError:
            # fallback for python-style lists with no quotes
            import ast
            pd_obj = ast.literal_eval(t)
        return Link(pd_obj)
    if _RE_XPD.search(t):
        try:
            return Link(t)
        except Exception:
            items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
            if not items:
                raise
            blocks = []
            for it in items:
                nums = [int(x.strip()) for x in it.split(',')]
                if len(nums) != 4:
                    raise ValueError("PD block must have 4 integers")
                blocks.append(nums)
            return Link(str(blocks))
    if (t.startswith("{") or t.startswith("[")) and not _RE_PDLIST.match(t):
        try:
            obj = json.loads(t)
            for key in ("pd","PD","pd_code","PD_code","dt","DT"):
                if key in obj:
                    return parse_link_strict(obj[key])
        except Exception:
            pass
    raise ValueError("Not a PD/DT code")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

# --------- 4) Spherogram helpers ----------
def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def can_reduce_basic(L: Link) -> bool:
    before = crossings(L)
    tmp = Link(L.PD_code())
    changed = tmp.simplify(mode="basic")
    return bool(changed and crossings(tmp) < before)

# --- UPDATED: pure RIII only (no RI/RII), in-place on the given Link ---
def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    """
    Apply up to k random Type-III moves using Spherogram internals
    without triggering any RI/RII simplification. Returns (link, done).
    """
    # lazy import so we don't change your global imports
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link  # operate in-place
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)  # in-place RIII
            if crossings(L) == c0:  # ensure it's a pure RIII (no crossing change)
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

# --------- 5) RL Environment (macro actions with blocking + simple ranking reward) ----------
# The action is MultiDiscrete: [mode, cap]
#   mode ∈ {0:basic, 1:level, 2:pickup, 3:backtrack}
#   cap  ∈ {0..cap_max}

maxstepsdone = 500

# 1) Config
@dataclass
class EnvCfg:
    max_steps: int = maxstepsdone
    step_penalty: float = 0.05           # small time cost
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    # Simple mode ranking reward: 0 > 1 > 2 > 3
    # mode_rewards: tuple[float, float, float, float] = (3.0, 2.0, 1.0, 0.0)
    # Reward shaping weights (dense progress)
    w_delta: float = 1.0          # reward per crossing removed
    w_uphill: float = 0.5         # extra penalty per crossing added (beyond losing delta)
    w_potential: float = 0.02     # small push toward low crossings

class SphKnotEnv(gym.Env):
    # ... keep everything else as you have it ...

    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)

        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )

        self.L: Optional[Link] = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False

        # NEW: blocking state (True = blocked). Index by mode.
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        """If requested mode is blocked, advance to the next unblocked mode (cyclic)."""
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        # Fallback (shouldn't happen): use backtrack
        return min(3, self.num_actions - 1)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        for _ in range(10):
            s = self.rng.choice(self.pd_lines)
            try:
                self.L = parse_link_strict(s)
                break
            except Exception:
                self.L = None
        if self.L is None:
            self.L = parse_link_strict(self.pd_lines[0])
        return self._obs(), {"crossings": crossings(self.L)}

    def step(self, action):
        # Unpack action
        self._steps += 1
        if isinstance(action, (list, tuple, np.ndarray)):
            mode_req, cap = int(action[0]), int(action[1])
        else:
            mode_req, cap = int(action), 0
        cap = max(0, min(cap, self.cfg.cap_max))

        # Apply blocking: remap to first unblocked mode (cyclic from requested)
        mode = self._map_blocked_mode(mode_req)

        # Context
        c_before = crossings(self.L)

        # Execute chosen (possibly remapped) mode
        if mode == 0:
            self.L.simplify(mode="basic")

        elif mode == 1:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="level", type_III_limit=steps)

        elif mode == 2:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="pickup", type_III_limit=steps)

        elif mode == 3 and self.num_actions == 4:
            steps = (cap if cap > 0 else 1)
            self.L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
            # optional: tiny RIII shuffle is ok to keep or remove; keeping it is fine
            self.L, _ = riii_shuffle_only_link(self.L, min(steps, 2))

        # Post
        c_after = crossings(self.L)
        delta = c_before - c_after  # success iff delta > 0
        self._last_drop = max(delta, 0)

        # --- Reward: simple ranking by mode, small step penalty
        # reward = self.cfg.mode_rewards[mode] - self.cfg.step_penalty
        # --- Reward: dense progress toward fewer crossings
        # delta = c_before - c_after  (positive is good)
        reward = (
            self.cfg.w_delta * delta
            - self.cfg.w_uphill * max(0, -delta)
            - self.cfg.w_potential * c_after
            - self.cfg.step_penalty
        )

        # Finish bonus (unchanged)
        done = False
        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            done = True
        if self._steps >= self.cfg.max_steps:
            done = True

        # --- Blocking logic
        if delta > 0:
            # success -> reset blocks
            self._reset_blocks()
        else:
          if mode == 3:
              self._reset_blocks()
          else:
              # only block if it actually got worse; allow 0-delta tries
              if delta < 0:
                  self._blocked[mode] = True
        # else:
        #     if mode == 3:
        #         # backtrack used -> reset to start of cycle
        #         self._reset_blocks()
        #     else:
        #         # unsuccessful non-backtrack -> block this mode
        #         self._blocked[mode] = True

        # After-backtrack flag (kept for your obs if needed)
        self._after_backtrack = (mode == 3 and self.num_actions == 4)

        info = {
            "crossings": c_after,
            "delta": delta,
            "mode_requested": mode_req,
            "mode_effective": mode,
            "cap": cap,
            "blocked": tuple(self._blocked),
        }
        return self._obs(), reward, done, False, info

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.logger import configure

# === Paths ===
BASE = "/content/drive/MyDrive/crossing-reduction-unknotting"
PD_PATH = f"{BASE}/3-16.txt"
SMALL_JONES = f"{BASE}/small-jones.txt"
BEST_MODEL_PATH = f"{BASE}/best_model.zip"   # flat, no subfolder
TB_DIR = f"{BASE}/tb_knots"                  # tensorboard logs here
LOCAL_RANDOM = f"{BASE}/random_diagrams.csv"
LOCAL_RANDOM2 = f"{BASE}/random_diagrams_2.csv"
LOCAL3 = f"{BASE}/very_hard_unknots_2.csv"
os.makedirs(TB_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# --------- 6) Load mixed CSVs from GCS (first column only), then shuffle ----------
# Files:
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"         # large file
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"
GCS_CSV_PATH_RANDOM = "gs://gdm-unknotting/random_diagrams.csv"    # much smaller file
GCS_CSV_PATH_RANDOM2 = "gs://gdm-unknotting/random_diagrams_2.csv"

# Parsing options (set per file if needed)
HAS_HEADER_MAIN = True
HAS_HEADER_VERY = True
HAS_HEADER_RANDOM = True
DELIMITER_MAIN  = None   # None = auto-sniff; else set "," or "\t"
DELIMITER_VERY  = None
DELIMITER_RANDOM  = None
MAX_ROWS_MAIN   = 385   # cap from the big file
SEED            = 42        # for reproducible shuffle

# 1) Read first column from all CSVs
raw_main = read_first_column_gcs(
    GCS_CSV_PATH_MAIN,
    max_lines=MAX_ROWS_MAIN,
    has_header=HAS_HEADER_MAIN,
    delimiter=DELIMITER_MAIN,
    encoding="utf-8",
)
raw_very = read_first_column_gcs(
    GCS_CSV_PATH_VERY,
    max_lines=None,                 # load the small file completely
    has_header=HAS_HEADER_VERY,
    delimiter=DELIMITER_VERY,
    encoding="utf-8",
)

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

raw_random = read_first_col_local(LOCAL_RANDOM, has_header=HAS_HEADER_RANDOM, encoding="utf-8")
raw_random2 = read_first_col_local(LOCAL_RANDOM2, has_header=HAS_HEADER_RANDOM, encoding="utf-8")
raw3 = read_first_col_local(LOCAL3, has_header=HAS_HEADER_RANDOM, encoding="utf-8")

print(f"[MAIN @ GCS] {len(raw_main)} rows (capped {MAX_ROWS_MAIN}).")
print(f"[VERY @ GCS] {len(raw_very)} rows.")
print(f"[RANDOM @ local] {len(raw_random)} rows.")
print(f"[RANDOM2 @ local] {len(raw_random2)} rows.")
print(f"[RAW3 @ local] {len(raw3)} rows.")

# 2) Concatenate (VERY_HARD first so they are guaranteed included), then clean to PD/DT
combined_raw = list(raw_very) + list(raw_main) + list(raw_random) + list(raw_random2) + list(raw3)
pd_lines = clean_pd_lines(combined_raw, max_keep=None)
assert pd_lines, "No valid PD/DT lines after cleaning—check the CSV columns or delimiters."

# 3) Shuffle for training (reproducible)
rng = random.Random(SEED)
rng.shuffle(pd_lines)

print(f"[Data] Kept {len(pd_lines)} PD/DT strings after strict filtering and shuffling.")
# (Optionally, deduplicate; uncomment if needed)
# pd_lines = list(dict.fromkeys(pd_lines))
# print(f"[Data] After de-duplication: {len(pd_lines)} PD/DT strings.")

#pd_lines_train = pd_lines[:TRAIN_ROWS]
#pd_lines_test = pd_lines[TRAIN_ROWS+1:MAX_ROWS]

def _obs_patch(self):
    c = crossings(self.L)
    try:
        comps = len(self.L.link_components)
    except Exception:
        comps = 1
    tmp = Link(self.L.PD_code())
    reduced = tmp.simplify(mode="basic")
    can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
    recent = 1.0 if getattr(self, "_last_drop", 0) > 0 else 0.0
    return np.array([c, comps, self._steps, can_reduce, recent, 1.0], dtype=np.float32)

# Monkey-patch if needed
SphKnotEnv._obs = getattr(SphKnotEnv, "_obs", _obs_patch)

# --------- 7) Build envs ----------
# (Optional) If you want backtrack available in this run, set allow_backtrack=True here:
cfg = EnvCfg(max_steps=maxstepsdone, allow_backtrack=True)  # flip to True later if helpful
def make_env():
    return SphKnotEnv(pd_lines, cfg)

env = make_env()
vec_env = DummyVecEnv([lambda: make_env()])

obs, info = env.reset()
print("[Env] obs:", obs, "start crossings:", info["crossings"])

[MAIN @ GCS] 385 rows (capped 385).
[VERY @ GCS] 385 rows.
[RANDOM @ local] 770 rows.
[RANDOM2 @ local] 770 rows.
[RAW3 @ local] 385 rows.
[Data] Kept 2695 PD/DT strings after strict filtering and shuffling.
[Env] obs: [35.  1.  0.  0.  0.  1.] start crossings: 35


In [ ]:
import os

# The BASE directory is defined in hnGBP8uScIAl
print(f"Listing contents of: {BASE}/")

# List all files and directories in the BASE path
if os.path.exists(BASE):
    for root, dirs, files in os.walk(BASE):
        # Only show immediate children of BASE for brevity unless user specifies more depth
        if root == BASE:
            print("  Directories:")
            for d in dirs:
                print(f"    - {d}/")
            print("  Files:")
            for f in files:
                print(f"    - {f}")
            break # Only list the top level of BASE for now
else:
    print(f"The directory '{BASE}' does not exist. Please check the BASE path.")


Listing contents of: /content/drive/MyDrive/crossing-reduction-unknotting/
  Directories:
    - tb_knots/
  Files:
    - connected_sums_allpairs_limit10.csv.gz
    - very_hard_unknots_2.csv
    - small-jones.txt
    - generated_pd_from_rl.txt
    - generated_pd_backtrack.txt
    - ppo_50000_steps.zip
    - ppo_100000_steps.zip
    - ppo_110000_steps.zip
    - best_model.zip
    - ppo_160000_steps.zip
    - generated_T_pd_backtrack.txt
    - ppo_knot_rl_spherogram_continued.zip
    - knotinfo_data_complete.xls
    - min_pd_results_flips1.jsonl
    - min_pd_results_flips1_jones_vec_knotinfo.jsonl
    - min_pd_results_flips1_jones_vec_knotinfo.txt
    - min_pd_results_flips1_jones.jsonl
    - knotinfo_match_results.jsonl
    - knotinfo_unknotting_candidates.jsonl
    - random_diagrams_2.csv
    - random_diagrams.csv


In [ ]:
# --------- 7) Train PPO ----------

# === Envs ===
vec_env  = DummyVecEnv([lambda: make_env()])
eval_env = DummyVecEnv([lambda: make_env()])

seed = random.randint(1, 100)
SEED = random.seed(seed)
TOTAL_STEPS = 12288*1

# === Load existing model or start fresh ===
if os.path.exists(BEST_MODEL_PATH):
    print(f"[Resume] Loading {BEST_MODEL_PATH}")
    model = PPO.load(BEST_MODEL_PATH, env=vec_env, device="auto")
    model.set_logger(configure(TB_DIR, ["stdout", "tensorboard"]))
else:
    print("[Fresh] Starting new PPO model")
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=lambda p: 3e-4 * p, # linear decay
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        tensorboard_log=TB_DIR,
        seed=SEED,
    )

# === Callbacks ===
# EvalCallback will now save directly as /MyDrive/crossing-reduction/best_model.zip
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=BASE,
    eval_freq=10000,
    n_eval_episodes=1,
    deterministic=True,
)
# (optional) periodic checkpoints — also in flat folder
ckpt_cb = CheckpointCallback(save_freq=50000, save_path=BASE, name_prefix="ppo")

# === Train (continues if loaded) ===
model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=[eval_cb, ckpt_cb],
    reset_num_timesteps=False,
    progress_bar=False, # Set to False to avoid LiveError in Colab
)

# === Save snapshot ===
final_path = f"{BASE}/ppo_knot_rl_spherogram_continued.zip"
model.save(final_path)
print(f"[Train] Saved -> {final_path}")

[Resume] Loading /content/drive/MyDrive/crossing-reduction-unknotting/best_model.zip
Logging to /content/drive/MyDrive/crossing-reduction-unknotting/tb_knots
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 403      |
|    ep_rew_mean     | -142     |
| time/              |          |
|    fps             | 149      |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 142048   |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 403           |
|    ep_rew_mean          | -142          |
| time/                   |               |
|    fps                  | 211           |
|    iterations           | 2             |
|    time_elapsed         | 19            |
|    total_timesteps      | 144096        |
| train/                  |               |
|    approx_kl            | 1.0681426e-05 |
|    clip_fraction

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=150000, episode_reward=-342.00 +/- 0.00
Episode length: 500.00 +/- 0.00
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 500          |
|    mean_reward          | -342         |
| time/                   |              |
|    total_timesteps      | 150000       |
| train/                  |              |
|    approx_kl            | 8.843665e-06 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.96        |
|    explained_variance   | 0.168        |
|    learning_rate        | 8.07e-06     |
|    loss                 | 30           |
|    n_updates            | 700          |
|    policy_gradient_loss | -0.000131    |
|    value_loss           | 66.8         |
------------------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 403      |
|    ep_rew

In [ ]:
OUT_DIR = BASE

In [ ]:
#K1 = [[1,9,2,8],[3,11,4,10],[5,13,6,12],[7,1,8,14],[9,3,10,2],[11,5,12,4],[13,7,14,6]] This is 7_1
#K2 = [[8,1,9,2],[10,3,11,4],[12,5,13,6],[14,7,1,8],[2,9,3,10],[4,11,5,12],[6,13,7,14]] This is the mirror of 7_1

#K1 = [[2,8,3,7],[4,10,5,9],[6,2,7,1],[8,4,9,3],[10,6,1,5]] # This is 5_1
#K2 = [[1,10,2,11],[3,13,4,12],[5,15,6,14],[7,1,8,16],[9,2,10,3],[11,9,12,8],[13,5,14,4],[15,7,16,6]] # This is 8_2

K1 = [[4,2,5,1],[8,6,1,5],[6,3,7,4],[2,7,3,8]] # This is 4_1
K2 = [[1,13,2,12],[3,11,4,10],[5,17,6,16],[7,15,8,14],[9,1,10,18],[11,3,12,2],[13,5,14,4],[15,7,16,6],[17,9,18,8]] # This is 9_10

# ---------- connected sum (your build uses .connected_sum) ----------
def connected_sum_pd(PD1, PD2, simplify=True):
    L1, L2 = Link(PD1), Link(PD2)
    out = L1.connected_sum(L2)  # returns new Link or mutates in place
    L = out if out is not None else L1
    if simplify and hasattr(L, "simplify"):
        try:
            L.simplify()
        except TypeError:
            pass
    return [list(q) for q in L.PD_code()]  # normalize to list-of-lists

T = connected_sum_pd(K1,K2)
T

[[11, 17, 12, 16],
 [15, 13, 16, 12],
 [13, 18, 14, 19],
 [17, 14, 18, 15],
 [19, 5, 20, 4],
 [21, 3, 22, 2],
 [23, 9, 24, 8],
 [25, 7, 0, 6],
 [1, 11, 2, 10],
 [3, 21, 4, 20],
 [5, 23, 6, 22],
 [7, 25, 8, 24],
 [9, 1, 10, 0]]

In [ ]:
# -----------------------------------------
# Randomize the knot T via backtrack + RIII
# -----------------------------------------

# Parameters (re-using from cell EhrCjVQukWfN for consistency)
NUM_VARIANTS_FOR_T = 100    # how many randomised diagrams for T
BACKTRACK_STEPS_MIN = 6    # min steps per randomisation
BACKTRACK_STEPS_MAX = 6   # max steps per randomisation
RIII_STEPS_MAX      = 25    # as in your env, min(steps, 2)
GEN_T_OUT_PATH      = f"{BASE}/generated_T_pd_backtrack.txt"

print("Randomising knot T:")

generated_T_pd_lines = []  # JSON PD list strings

# T is already a list of quads, convert to Link object
L0_T = Link(T)

for v in range(NUM_VARIANTS_FOR_T):
    # fresh copy from PD code so we don't mutate L0_T
    pd0 = L0_T.PD_code()
    # handle both list and PDCode cases
    if isinstance(pd0, list):
        L = Link(pd0)
    else:
        L = Link([[int(getattr(e, "label", e)) for e in vtx] for vtx in pd0])

    # choose random number of backtrack steps
    steps = random.randint(BACKTRACK_STEPS_MIN, BACKTRACK_STEPS_MAX)

    # your usual "add crossings and shuffle" move:
    L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
    L, _ = riii_shuffle_only_link(L, min(steps, RIII_STEPS_MAX))

    # now extract PD as list-of-quads
    pd_obj = L.PD_code()
    if isinstance(pd_obj, list):
        quads = [[int(x) for x in quad] for quad in pd_obj]
    else:
        quads = [[int(getattr(edge, "label", edge)) for edge in vtx] for vtx in pd_obj]

    pd_str_new = json.dumps(quads)
    generated_T_pd_lines.append(pd_str_new)
    print(f"  variant {v:02d}: steps={steps}, crossings_final={len(quads)}")

print(f"\nGenerated {len(generated_T_pd_lines)} randomised PDs from T.")

# -----------------------------------------
# Save new PDs to a file (JSON PD list format)
# -----------------------------------------
with open(GEN_T_OUT_PATH, "w") as f:
    for line in generated_T_pd_lines:
        f.write(line + "\n")

print(f"Wrote {len(generated_T_pd_lines)} PD lines for T to {GEN_T_OUT_PATH}")

# Optional: peek at the generated PDs for T
print(f"\nPeeking at the first {min(NUM_VARIANTS_FOR_T, len(generated_T_pd_lines))} generated PDs for T:\n")
for i, pd_str in enumerate(generated_T_pd_lines[:NUM_VARIANTS_FOR_T]):
    quads = json.loads(pd_str)
    try:
        L = Link(quads)
        c = len(L.crossings)
    except Exception as e:
        L = None
        c = "?"
    print(f"[{i}] crossings = {c}")
    print(f"    {pd_str}")

Randomising knot T:
  variant 00: steps=6, crossings_final=24
  variant 01: steps=6, crossings_final=21
  variant 02: steps=6, crossings_final=23
  variant 03: steps=6, crossings_final=22
  variant 04: steps=6, crossings_final=25
  variant 05: steps=6, crossings_final=23
  variant 06: steps=6, crossings_final=23
  variant 07: steps=6, crossings_final=23
  variant 08: steps=6, crossings_final=20
  variant 09: steps=6, crossings_final=23
  variant 10: steps=6, crossings_final=25
  variant 11: steps=6, crossings_final=25
  variant 12: steps=6, crossings_final=23
  variant 13: steps=6, crossings_final=21
  variant 14: steps=6, crossings_final=25
  variant 15: steps=6, crossings_final=22
  variant 16: steps=6, crossings_final=25
  variant 17: steps=6, crossings_final=23
  variant 18: steps=6, crossings_final=22
  variant 19: steps=6, crossings_final=23
  variant 20: steps=6, crossings_final=24
  variant 21: steps=6, crossings_final=24
  variant 22: steps=6, crossings_final=24
  variant 23: 

In [ ]:
PD_PATH2 = GEN_T_OUT_PATH

In [ ]:
# ================================# 1) Read first PDs from PD_PATH2# ================================
def read_pd_lines_from_file(path: str, max_lines: int | None = None):
    """Read non-empty lines from a local PD file and return the first `max_lines`."""
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:max_lines]

def parse_pd_line_to_list(line: str):
    """
    Parse either
      - JSON PD lists: '[[1,5,2,4],[3,1,4,6],...]'
      - or PD[X[1,5,2,4], X[3,1,4,6], ...]
    into [[a,b,c,d], ...].
    """
    t = line.strip()

    # 1) Try JSON / Python list first
    try:
        obj = json.loads(t)
        if isinstance(obj, list) and obj and isinstance(obj[0], (list, tuple)):
            quads = []
            for quad in obj:
                if len(quad) != 4:
                    raise ValueError(f"PD block must have 4 integers, got {quad!r}")
                quads.append([int(x) for x in quad])
            return quads
    except Exception:
        pass  # not JSON, fall back to PD[X[...]] format

    # 2) Fallback: PD[X[...], X[...], ...] format
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
    if not items:
        raise ValueError(f"No X[...] blocks found in line: {line!r}")

    quads = []
    for it in items:
        nums = [int(x.strip()) for x in it.split(',')]
        if len(nums) != 4:
            raise ValueError(f"PD block must have 4 integers, got {nums!r}")
        quads.append(nums)
    return quads

# Assuming PD_PATH2 is defined elsewhere, e.g., in an earlier cell
# PD_PATH2 = f"{BASE}/generated_T_pd_backtrack.txt"
raw_pd_lines = read_pd_lines_from_file(PD_PATH2, max_lines=NUM_VARIANTS_FOR_T)
print("First PD lines from PD_PATH2:")
for i, s in enumerate(raw_pd_lines):
    print(f"  [{i}] {s}")

pd_lists = [parse_pd_line_to_list(s) for s in raw_pd_lines]
print("\nParsed as PD lists (lengths):", [len(p) for p in pd_lists])

# ================================# 2) Crossing flip + PD variants# ================================
def flip_crossing_quad(quad):
    """[a,b,c,d] -> [b,c,d,a]"""
    a, b, c, d = quad
    return [b, c, d, a]

def pd_list_to_str(pd_list):
    """
    Stringify to something parse_link_strict accepts.
    This produces '[[1,5,2,4],[3,1,4,6],...]' which matches the
    _RE_PDLIST branch in parse_link_strict.
    """
    import json # Ensure json is imported if not globally available
    return json.dumps(pd_list)

def generate_changed_pds(pd_list, k):
    """
    Yield all PD-lists obtained by flipping exactly k crossings,
    in lexicographic order on crossing indices.
    """
    n = len(pd_list)
    for idxs in itertools.combinations(range(n), k):
        idxs_set = set(idxs)
        new_pd = []
        for i, quad in enumerate(pd_list):
            if i in idxs_set:
                new_pd.append(flip_crossing_quad(quad))
            else:
                new_pd.append(list(quad))
        yield idxs, new_pd

# ================================# 3) RL wrapper: run agent on one PD# ================================
# Load your best model (avoid reloading if already in memory)
try:
    model
except NameError:
    model = PPO.load(BEST_MODEL_PATH, device="cpu")
    print("\n[Model] Loaded PPO model from:", BEST_MODEL_PATH)

def make_single_env(pd_list, cfg: EnvCfg):
    """
    Build a DummyVecEnv whose SphKnotEnv always starts from this single PD.
    """
    pd_str = pd_list_to_str(pd_list)
    pd_lines_single = [pd_str]

    def _make():
        return SphKnotEnv(pd_lines_single, cfg)

    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg: EnvCfg,
                        episodes: int = 3, label: str = "",
                        return_best_pd: bool = False):
    """
    Run the PPO agent on a single PD (wrapped into a single-knot env).

    Default return (backwards-compatible):
        (success: bool, best_crossings: int)

    If return_best_pd=True, returns:
        (success: bool, best_crossings: int, best_pd_list: list[list[int]])

    Here, best_pd_list is the *best* (fewest crossings) PD encountered during the rollouts.
    """
    vec_env = make_single_env(pd_list, cfg)

    success = False
    best_crossings_global = len(pd_list)
    best_pd_global = [list(q) for q in pd_list]

    for ep in range(episodes):
        obs = vec_env.reset()  # no info at vec level

        # Track best within this episode
        best_crossings_ep = best_crossings_global
        best_pd_ep = best_pd_global

        for step in range(cfg.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)

            info = infos[0]
            cr = info.get("crossings", None)

            # Pull current PD directly from the underlying env (DummyVecEnv → envs[0])
            try:
                cur_pd = [list(q) for q in vec_env.envs[0].L.PD_code()]
            except Exception:
                cur_pd = None

            if cr is not None and cr < best_crossings_ep and cur_pd is not None:
                best_crossings_ep = cr
                best_pd_ep = cur_pd

            # success criterion: crossings == 0
            if cr == 0:
                success = True
                if cur_pd is not None:
                    best_crossings_ep = 0
                    best_pd_ep = cur_pd
                break

            if dones[0]:
                break

        # Update global best across episodes
        if best_crossings_ep < best_crossings_global:
            best_crossings_global = best_crossings_ep
            best_pd_global = best_pd_ep

        if success:
            break

    vec_env.close()

    if return_best_pd:
        return success, best_crossings_global, best_pd_global
    return success, best_crossings_global

# ================================# 3b) Helpers: (optional) extra simplify + SnapPy identification# ================================
def simplify_pd_basic(pd_list, rounds: int = 5):
    """Run a few rounds of Spherogram basic simplification and return a (possibly) smaller PD."""
    try:
        L = Link(pd_list)
        for _ in range(rounds):
            before = len(L.crossings)
            try:
                L.simplify(mode="basic")
            except TypeError:
                L.simplify()
            after = len(L.crossings)
            if after >= before:
                break
        return [list(q) for q in L.PD_code()]
    except Exception:
        # If anything goes wrong, just return the input
        return [list(q) for q in pd_list]

def identify_knot_snappy(pd_list):
    """Try to identify the knot/link represented by pd_list using SnapPy."""
    # Try via Spherogram → exterior() (preferred when available)
    try:
        L = Link(pd_list)
        M = L.exterior()
        try:
            name = M.identify()
        except Exception:
            name = None
        return name, M
    except Exception:
        pass

    # Fallback: SnapPy Link directly
    try:
        Ls = snappy.Link(pd_list)
        M = Ls.exterior()
        try:
            name = M.identify()
        except Exception:
            name = None
        return name, M
    except Exception as e:
        return None, None

# ================================# 4) Untangling number search# ================================
def compute_untangling_number(pd_list,
                              model,
                              cfg: EnvCfg,
                              max_changes: int | None = None,
                              episodes: int = 3):
    """
    Search for minimal k \u2265 1 such that flipping k crossings
    yields a diagram which the RL machine can unknot.

    Returns (k, idxs) where
      - k is the untangling number (int or None)
      - idxs is the tuple of crossing indices to flip (or None)
    """
    n = len(pd_list)
    if max_changes is None:
        max_changes = n

    for k in range(1, max_changes + 1):
        for idxs, new_pd in generate_changed_pds(pd_list, k):
            success, _ = run_unknotter_on_pd(new_pd, model, cfg, episodes=episodes)
            if success:
                return k, idxs

    return None, None


First PD lines from PD_PATH2:
  [0] [[25, 21, 26, 20], [21, 25, 22, 24], [23, 18, 24, 19], [19, 22, 20, 23], [15, 41, 16, 40], [13, 45, 14, 44], [9, 29, 10, 28], [1, 33, 2, 32], [45, 27, 46, 26], [43, 15, 44, 14], [37, 11, 38, 10], [29, 7, 30, 6], [27, 1, 28, 0], [42, 17, 43, 18], [41, 17, 42, 16], [36, 5, 37, 6], [35, 5, 36, 4], [30, 7, 31, 8], [31, 9, 32, 8], [2, 33, 3, 34], [3, 35, 4, 34], [38, 11, 39, 12], [39, 13, 40, 12], [46, 0, 47, 47]]
  [1] [[25, 19, 26, 18], [19, 25, 20, 24], [21, 16, 22, 17], [17, 20, 18, 21], [15, 37, 16, 36], [9, 41, 10, 40], [5, 31, 6, 30], [1, 33, 2, 32], [41, 27, 0, 26], [37, 13, 38, 12], [35, 9, 36, 8], [31, 5, 32, 4], [27, 1, 28, 0], [34, 7, 35, 8], [33, 7, 34, 6], [29, 29, 30, 28], [14, 14, 15, 13], [23, 23, 24, 22], [11, 39, 12, 38], [10, 39, 11, 40], [3, 3, 4, 2]]
  [2] [[27, 17, 28, 16], [17, 25, 18, 24], [21, 14, 22, 15], [15, 20, 16, 21], [13, 37, 14, 36], [7, 45, 8, 44], [5, 31, 6, 30], [1, 35, 2, 34], [45, 29, 0, 28], [37, 9, 38, 8], [35, 7, 

In [ ]:
original_pd_list = f"{BASE}/generated_T_pd_backtrack.txt"  # file with generated JSON PD lists

In [ ]:
import itertools

# --- ANSI colors (safe defaults) ---
RESET = "\033[0m"
RED   = "\033[91m"
GREEN = "\033[92m"
YELLOW= "\033[93m"

# Require OUT_DIR to be defined by your Drive/paths cell
import os, json
if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR is not defined. Please run the Drive/paths cell first.")

flips = 1  # Number of crossings to flip (now: exactly one)

all_possible_modified_knots_info = []

# Assuming pd_lists contains the original PD lists to work with
for original_pd_list in pd_lists:
    n_crossings = len(original_pd_list)

    # Ensure there are enough crossings to perform the requested number of flips
    if n_crossings < flips:
        print(f"Skipping PD list with only {n_crossings} crossings (less than {flips} required flips).")
        continue

    # Generate all unique combinations of 'flips' distinct crossing indices
    for flipped_indices_combo in itertools.combinations(range(n_crossings), flips):
        flipped_indices = sorted(list(flipped_indices_combo))  # Ensure sorted for consistent reporting

        flipped_pd_list = []
        for i, quad in enumerate(original_pd_list):
            if i in flipped_indices:
                flipped_pd_list.append(flip_crossing_quad(quad))
            else:
                flipped_pd_list.append(list(quad))  # Ensure a deep copy

        all_possible_modified_knots_info.append({
            "original_pd_list": original_pd_list,
            "flipped_pd_list": flipped_pd_list,
            "flipped_indices": tuple(flipped_indices)
        })

print(f"Generated {len(all_possible_modified_knots_info)} all possible modified knot variants (combinations of {flips} flips).")

print("\nEvaluating RL model on all possible modified knots:")

results_summary_all_possible = []

for i, knot_info in enumerate(all_possible_modified_knots_info):
    original_pd_list = knot_info["original_pd_list"]
    flipped_pd_list = knot_info["flipped_pd_list"]
    flipped_indices = knot_info["flipped_indices"]

    initial_crossings = len(original_pd_list)

    print(f"\n--- Knot variant {i+1}/{len(all_possible_modified_knots_info)} ---")
    print(f"  Original crossings: {initial_crossings}")
    print(f"  Flipped indices: {flipped_indices}")

    # Run the unknotter on the flipped PD list (and keep the best PD it finds)
    success, min_crossings_found, best_pd_list = run_unknotter_on_pd(
        flipped_pd_list,
        model,
        cfg,
        episodes=1,  # keep fast for evaluation
        label=f"Knot {i+1} (flips {flipped_indices})",
        return_best_pd=True,
    )

    print(f"  Min crossings found: {min_crossings_found}")

    results_summary_all_possible.append({
        "knot_index": i + 1,
        "original_crossings": initial_crossings,
        "flipped_indices": flipped_indices,
        "rl_unknot_success": success,
        "min_crossings_found": min_crossings_found,
        "best_pd_list": best_pd_list,
        "flipped_pd_list": flipped_pd_list,
    })

print("\n=================== Final Summary (All Possible Flips) ===================")
for res in results_summary_all_possible:
    status_color = RED if res["rl_unknot_success"] else RESET
    print(
        f"{status_color}Knot {res['knot_index']}: "
        f"Original Crossings={res['original_crossings']}, "
        f"Flipped Indices={res['flipped_indices']}, "
        f"RL Unknot Success={res['rl_unknot_success']}, "
        f"Min Crossings Found={res['min_crossings_found']}"
        f"{RESET}"
    )
print("========================================================================")

# ============================
# Write minimal-PD outputs to file (for later diagnosis)
# ============================
os.makedirs(OUT_DIR, exist_ok=True)

# Deterministic name (overwrite on rerun)
out_path = os.path.join(OUT_DIR, f"min_pd_results_flips{flips}.jsonl")

with open(out_path, "w") as f:
    for res in results_summary_all_possible:
        rec = dict(res)
        rec["flipped_indices"] = list(rec["flipped_indices"])  # JSON-friendly
        f.write(json.dumps(rec) + "\n")

print(f"\nWrote minimal-PD results to: {out_path}")

Streaming output truncated to the last 5000 lines.

--- Knot variant 1750/2290 ---
  Original crossings: 25
  Flipped indices: (13,)
  Min crossings found: 15

--- Knot variant 1751/2290 ---
  Original crossings: 25
  Flipped indices: (14,)
  Min crossings found: 13

--- Knot variant 1752/2290 ---
  Original crossings: 25
  Flipped indices: (15,)
  Min crossings found: 9

--- Knot variant 1753/2290 ---
  Original crossings: 25
  Flipped indices: (16,)
  Min crossings found: 15

--- Knot variant 1754/2290 ---
  Original crossings: 25
  Flipped indices: (17,)
  Min crossings found: 15

--- Knot variant 1755/2290 ---
  Original crossings: 25
  Flipped indices: (18,)
  Min crossings found: 11

--- Knot variant 1756/2290 ---
  Original crossings: 25
  Flipped indices: (19,)
  Min crossings found: 11

--- Knot variant 1757/2290 ---
  Original crossings: 25
  Flipped indices: (20,)
  Min crossings found: 15

--- Knot variant 1758/2290 ---
  Original crossings: 25
  Flipped indices: (21,)
  Mi

In [ ]:
# ============================================================
# Compute Jones polynomial from RL-minimal PDs (pure python)
# Writes: OUT_DIR/min_pd_results_flips{flips}_jones.jsonl
# Defines: results (list of dicts), for downstream cells
# ============================================================

import os, json
from fractions import Fraction

# --- require OUT_DIR + flips (as in your notebook conventions) ---
if "OUT_DIR" not in globals():
    raise NameError("OUT_DIR is not defined. Run your Drive/paths cell first.")
if "flips" not in globals():
    raise NameError("flips is not defined. Run the 'import itertools' cell first.")

in_path = os.path.join(OUT_DIR, f"min_pd_results_flips{flips}.jsonl")
if not os.path.exists(in_path):
    raise FileNotFoundError(f"Missing input JSONL: {in_path}. Run the 'import itertools' cell first.")

# -------------------------
# Laurent polynomial in A: dict exponent(int) -> coeff(int)
# -------------------------
def poly_add(p, q):
    r = dict(p)
    for e, c in q.items():
        r[e] = r.get(e, 0) + c
        if r[e] == 0:
            del r[e]
    return r

def poly_mul(p, q):
    r = {}
    for e1, c1 in p.items():
        for e2, c2 in q.items():
            e = e1 + e2
            r[e] = r.get(e, 0) + c1 * c2
    return {e:c for e,c in r.items() if c != 0}

def poly_monom(e, c=1):
    return {e: c} if c != 0 else {}

def poly_scale(p, k):
    if k == 0:
        return {}
    return {e: c*k for e,c in p.items()}

# -------------------------
# Union-Find for loop counting in each state
# -------------------------
class DSU:
    def __init__(self):
        self.parent = {}
        self.rank = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0
            return x
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
    def n_components(self):
        roots = set(self.find(x) for x in self.parent.keys())
        return len(roots)

# -------------------------
# Kauffman bracket from PD
# Convention:
#   A-smoothing pairs (a,b) and (c,d)
#   B-smoothing pairs (b,c) and (d,a)
# -------------------------
def bracket_from_pd(pd):
    n = len(pd)
    # Delta = -A^2 - A^-2
    Delta = poly_add(poly_scale(poly_monom(2), -1), poly_scale(poly_monom(-2), -1))

    labels = set()
    for (a,b,c,d) in pd:
        labels.update([a,b,c,d])

    total = {}

    for mask in range(1 << n):
        dsu = DSU()
        for x in labels:
            dsu.find(x)

        a_count = 0
        b_count = 0

        for i, (a,b,c,d) in enumerate(pd):
            if ((mask >> i) & 1) == 0:
                a_count += 1
                dsu.union(a, b)
                dsu.union(c, d)
            else:
                b_count += 1
                dsu.union(b, c)
                dsu.union(d, a)

        loops = dsu.n_components()
        mon = poly_monom(a_count - b_count, 1)

        # Delta^(loops-1)
        factor = {0: 1}
        for _ in range(loops - 1):
            factor = poly_mul(factor, Delta)

        total = poly_add(total, poly_mul(mon, factor))

    return total

# -------------------------
# Writhe heuristic (only affects an overall monomial shift)
# You told me you don't care about shift; still, keep it consistent.
# -------------------------
def crossing_sign_pd(quad):
    a,b,c,d = quad
    return 1 if (a < c) == (b < d) else -1

# -------------------------
# Convert bracket(A) to a Jones-like polynomial in variable t with integer exponents
# Using the usual normalization V = (-A)^(-3w) <D>, then substitute A = t^(-1/4).
# In typical PD conventions from the same generator, exponents land in Z.
# -------------------------
def jones_string_from_pd(pd_list):
    if not pd_list:
        return None, "Empty PD"

    pd = [tuple(map(int, q)) for q in pd_list]

    try:
        br = bracket_from_pd(pd)
        w = sum(crossing_sign_pd(q) for q in pd)

        # multiply by (-A)^(-3w) = (-1)^(-3w) * A^(-3w)
        sign = -1 if ((-3*w) % 2) else 1
        norm = poly_scale(poly_monom(-3*w, 1), sign)
        normed = poly_mul(norm, br)

        # Substitute A = t^(-1/4): A^e -> t^(-e/4)
        jt = {}
        for eA, c in normed.items():
            eT = Fraction(-eA, 4)
            jt[eT] = jt.get(eT, 0) + c
        jt = {e:c for e,c in jt.items() if c != 0}

        # If exponents are not integers, we still print them (rare in your pipeline)
        def exp_to_str(fr):
            if fr == 0:
                return ""
            if fr.denominator == 1:
                k = fr.numerator
                if k == 1:
                    return "t"
                return f"t^{k}"
            return f"t^({fr.numerator}/{fr.denominator})"

        # build string with descending exponent
        terms = []
        for e in sorted(jt.keys(), reverse=True):
            c = jt[e]
            mon = exp_to_str(e)
            if mon == "":
                term = f"{c}"
            else:
                if c == 1:
                    term = mon
                elif c == -1:
                    term = "-" + mon
                else:
                    term = f"{c}*{mon}"
            terms.append(term)

        if not terms:
            return "0", None

        s = terms[0]
        for t in terms[1:]:
            if t.startswith("-"):
                s += " - " + t[1:]
            else:
                s += " + " + t

        return s, None

    except Exception as e:
        return None, repr(e)

# -------------------------
# Load RL-minimal PDs, compute Jones, write deterministic JSONL
# -------------------------
results = []
with open(in_path, "r") as f:
    for line in f:
        rec = json.loads(line)
        best_pd = rec.get("best_pd_list")
        jstr, jerr = jones_string_from_pd(best_pd)

        results.append({
            "knot_index": rec.get("knot_index"),
            "flipped_indices": rec.get("flipped_indices"),
            "rl_unknot_success": rec.get("rl_unknot_success"),
            "min_crossings_found": rec.get("min_crossings_found"),
            "best_pd_crossings": len(best_pd) if best_pd else None,
            "jones": jstr,
            "jones_error": jerr,
        })

out_jones = os.path.join(OUT_DIR, f"min_pd_results_flips{flips}_jones.jsonl")
with open(out_jones, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"Loaded {len(results)} PDs from: {in_path}")
print(f"Wrote Jones results to:        {out_jones}")

# quick preview
for r in results[:5]:
    print(f"variant {r['knot_index']}: flips={r['flipped_indices']} bestX={r['best_pd_crossings']}  Jones={r['jones']}")

Loaded 2290 PDs from: /content/drive/MyDrive/crossing-reduction-unknotting/min_pd_results_flips1.jsonl
Wrote Jones results to:        /content/drive/MyDrive/crossing-reduction-unknotting/min_pd_results_flips1_jones.jsonl
variant 1: flips=[0] bestX=9  Jones=-t^-1 + t^-2 - 3*t^-3 + 5*t^-4 - 5*t^-5 + 6*t^-6 - 5*t^-7 + 4*t^-8 - 2*t^-9 + t^-10
variant 2: flips=[1] bestX=9  Jones=-t^-1 + t^-2 - 3*t^-3 + 5*t^-4 - 5*t^-5 + 6*t^-6 - 5*t^-7 + 4*t^-8 - 2*t^-9 + t^-10
variant 3: flips=[2] bestX=9  Jones=-t^-1 + t^-2 - 3*t^-3 + 5*t^-4 - 5*t^-5 + 6*t^-6 - 5*t^-7 + 4*t^-8 - 2*t^-9 + t^-10
variant 4: flips=[3] bestX=9  Jones=-t^-1 + t^-2 - 3*t^-3 + 5*t^-4 - 5*t^-5 + 6*t^-6 - 5*t^-7 + 4*t^-8 - 2*t^-9 + t^-10
variant 5: flips=[4] bestX=11  Jones=-t^2 + 2*t - 4 + 7*t^-1 - 9*t^-2 + 10*t^-3 - 10*t^-4 + 9*t^-5 - 6*t^-6 + 4*t^-7 - 2*t^-8 + t^-9


In [ ]:
# ============================================================
# Patch: output KnotInfo-compatible Jones vectors
# Format: [min_exp, max_exp, c_min, c_{min+1}, ..., c_max]
# where exponents are for the Jones polynomial variable (whatever you're using),
# and coefficients include internal zeros.
#
# Works from your existing `results` list where each item has:
#   - "jones" as a string like "-t^-1 + t^-2 - 3*t^-3 + ..."
#   - plus metadata (knot_index, flipped_indices, best_pd_crossings, ...)
#
# Produces:
#   - vectors_ki: list of records with "jones_vec_knotinfo"
#   - writes JSONL + TXT (one vector per line) next to out_path
# ============================================================

import re, json, os, glob

# Fallback OUT_DIR if the Drive-paths cell hasn't run yet
if "OUT_DIR" not in globals():
    OUT_DIR = "/content"  # safe local fallback
    # If running on Colab and Drive is mounted, prefer your project folder
    drive_proj = "/content/drive/MyDrive/crossing-reduction-unknotting"
    if os.path.exists("/content/drive") and os.path.isdir(drive_proj):
        OUT_DIR = drive_proj
    print("OUT_DIR was not defined yet; using:", OUT_DIR)

# If `results` isn't defined in-memory, reconstruct it from a saved Jones JSONL
if "results" not in globals():
    # Try the deterministic name first if flips exists
    candidates = []
    if "flips" in globals():
        candidates.append(os.path.join(OUT_DIR, f"min_pd_results_flips{flips}_jones.jsonl"))

    # Otherwise, search for any Jones JSONL in OUT_DIR
    candidates += sorted(glob.glob(os.path.join(OUT_DIR, "min_pd_results_flips*_jones.jsonl")))

    if not candidates:
        raise FileNotFoundError(
            "Could not find any '*_jones.jsonl' file to load results from.\n"
            f"Searched in: {OUT_DIR}\n"
            "Run the Jones computation cell first (the one that writes *_jones.jsonl), "
            "or set OUT_DIR to the folder containing that file."
        )

    jones_jsonl = candidates[-1]  # pick the last one (usually latest / highest flips)
    results = []
    with open(jones_jsonl, "r") as f:
        for line in f:
            results.append(json.loads(line))
    print(f"Loaded results from {jones_jsonl} ({len(results)} records)")

_int_pat = re.compile(r"-?\d+")

def parse_jones_string_to_dict(jstr):
    """
    Parse: "-t^-1 + t^-2 - 3*t^-3 + 5*t^-4 ..." -> {exp:int coeff}
    Exponents must be integers.
    """
    if jstr is None:
        return None
    s = str(jstr).strip()
    if s == "" or s == "0":
        return {}

    s = s.replace(" - ", " + -")
    parts = [p.strip() for p in s.split(" + ") if p.strip()]

    poly = {}
    for term in parts:
        term = term.replace(" ", "")
        if term == "":
            continue

        # coefficient and monomial split
        if "*t" in term:
            c_str, mon = term.split("*", 1)
            coeff = int(c_str)
        elif term.startswith("t") or term.startswith("-t"):
            coeff = -1 if term.startswith("-t") else 1
            mon = term[1:] if term.startswith("-t") else term
        else:
            # constant
            coeff = int(term)
            exp = 0
            poly[exp] = poly.get(exp, 0) + coeff
            continue

        # parse exponent from mon (starts with 't')
        if mon == "t":
            exp = 1
        elif mon.startswith("t^"):
            exp = int(mon[2:])  # handles negative too
        else:
            raise ValueError(f"Unrecognized monomial format: {mon} (from term {term})")

        poly[exp] = poly.get(exp, 0) + coeff

    return {e: c for e, c in poly.items() if c != 0}

def poly_dict_to_knotinfo_vector(poly):
    """
    Convert dict exp->coeff to KnotInfo vector [min_exp, max_exp, coeffs...]
    filling internal zeros. If poly is empty -> [0,0,0].
    """
    if poly is None:
        return None
    if len(poly) == 0:
        return [0, 0, 0]

    min_exp = min(poly.keys())
    max_exp = max(poly.keys())
    coeffs = []
    for e in range(min_exp, max_exp + 1):
        coeffs.append(int(poly.get(e, 0)))
    return [int(min_exp), int(max_exp)] + coeffs

# Build KnotInfo-compatible vectors for every result
vectors_ki = []
for r in results:
    jstr = r.get("jones")
    try:
        poly = parse_jones_string_to_dict(jstr) if jstr is not None else None
        vec_ki = poly_dict_to_knotinfo_vector(poly) if poly is not None else None
        err = None
    except Exception as e:
        vec_ki = None
        err = repr(e)

    vectors_ki.append({
        "knot_index": r.get("knot_index"),
        "flipped_indices": r.get("flipped_indices"),
        "best_pd_crossings": r.get("best_pd_crossings"),
        "jones_vec_knotinfo": vec_ki,
        "jones_vec_error": err or r.get("jones_error"),
    })

# Print a compact preview
print("=== KnotInfo-compatible Jones vectors ===")
for v in vectors_ki:
    print(f"variant {v['knot_index']:>3} flips={v['flipped_indices']} bestX={v['best_pd_crossings']}  vec={v['jones_vec_knotinfo']}")

# Save (1) JSONL with metadata
out_path_ki_jsonl = out_path.replace(".jsonl", "_jones_vec_knotinfo.jsonl")
with open(out_path_ki_jsonl, "w") as f:
    for v in vectors_ki:
        f.write(json.dumps(v) + "\n")
print("\nWrote KnotInfo-compatible vectors (JSONL) to:", out_path_ki_jsonl)

# Save (2) TXT with just the vector per line (easy to grep / load)
out_path_ki_txt = out_path.replace(".jsonl", "_jones_vec_knotinfo.txt")
with open(out_path_ki_txt, "w") as f:
    for v in vectors_ki:
        f.write(str(v["jones_vec_knotinfo"]) + "\n")
print("Wrote KnotInfo-compatible vectors (TXT) to:", out_path_ki_txt)

=== KnotInfo-compatible Jones vectors ===
variant   1 flips=[0] bestX=9  vec=[-10, -1, 1, -2, 4, -5, 6, -5, 5, -3, 1, -1]
variant   2 flips=[1] bestX=9  vec=[-10, -1, 1, -2, 4, -5, 6, -5, 5, -3, 1, -1]
variant   3 flips=[2] bestX=9  vec=[-10, -1, 1, -2, 4, -5, 6, -5, 5, -3, 1, -1]
variant   4 flips=[3] bestX=9  vec=[-10, -1, 1, -2, 4, -5, 6, -5, 5, -3, 1, -1]
variant   5 flips=[4] bestX=11  vec=[-9, 2, 1, -2, 4, -6, 9, -10, 10, -9, 7, -4, 2, -1]
variant   6 flips=[5] bestX=11  vec=[-9, 2, 1, -2, 4, -6, 9, -10, 10, -9, 7, -4, 2, -1]
variant   7 flips=[6] bestX=11  vec=[-9, 2, 1, -2, 4, -6, 9, -10, 10, -9, 7, -4, 2, -1]
variant   8 flips=[7] bestX=11  vec=[-9, 2, 1, -2, 4, -6, 9, -10, 10, -9, 7, -4, 2, -1]
variant   9 flips=[8] bestX=11  vec=[-10, 1, 1, -3, 6, -8, 11, -12, 11, -9, 7, -4, 2, -1]
variant  10 flips=[9] bestX=11  vec=[-9, 2, 1, -2, 4, -6, 9, -10, 10, -9, 7, -4, 2, -1]
variant  11 flips=[10] bestX=11  vec=[-10, 1, 1, -3, 6, -8, 11, -12, 11, -9, 7, -4, 2, -1]
variant  12 flips

In [ ]:
# ============================================================
# FINAL: Match Jones vectors against KnotInfo (Run-all safe)
#   - Accepts only vectors of format [min,max,c0,...,cN] (always!)
#   - Matches up to:
#       * q-shift (ignored by comparing coeffs only)
#       * mirror (reverse coeffs)
#     with extra constraint:
#       * abs(min-max) must match between input and KnotInfo
# ============================================================

import os, re, json, ast, glob
import pandas as pd
from collections import defaultdict

# ----------------------------
# Ensure PROJECT_DIR / OUT_DIR exist (self-contained)
# ----------------------------
if "PROJECT_DIR" not in globals() or "OUT_DIR" not in globals():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(
            "Could not mount Google Drive automatically. "
            "If you're not in Colab, set PROJECT_DIR/OUT_DIR manually.\n"
            f"Error: {e!r}"
        )

    PROJECT_DIR = "/content/drive/MyDrive/crossing-reduction-unknotting"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    OUT_DIR = PROJECT_DIR
    print("Initialized PROJECT_DIR =", PROJECT_DIR)
    print("Initialized OUT_DIR     =", OUT_DIR)

# ----------------------------
# Paths (Drive only, deterministic)
# ----------------------------
KNOTINFO_XLS_PATH = os.path.join(PROJECT_DIR, "knotinfo_data_complete.xls")
if not os.path.exists(KNOTINFO_XLS_PATH):
    raise FileNotFoundError(
        f"Missing KnotInfo file at:\n  {KNOTINFO_XLS_PATH}\n"
        "Please put 'knotinfo_data_complete.xls' into your Drive folder:\n"
        f"  {PROJECT_DIR}"
    )

# Auto-pick the newest/last vectors file in OUT_DIR
cands = sorted(glob.glob(os.path.join(OUT_DIR, "min_pd_results_flips*_jones_vec_knotinfo.txt")))
if not cands:
    raise FileNotFoundError(
        "Could not find any '*_jones_vec_knotinfo.txt' in OUT_DIR.\n"
        f"OUT_DIR = {OUT_DIR}\n"
        "Run the cell that writes KnotInfo-compatible Jones vectors first."
    )
USER_VECTORS_PATH = cands[-1]

print("KNOTINFO_XLS_PATH =", KNOTINFO_XLS_PATH)
print("USER_VECTORS_PATH =", USER_VECTORS_PATH)

# ----------------------------
# Read KnotInfo .xls (install xlrd if needed)
# ----------------------------
try:
    df = pd.read_excel(KNOTINFO_XLS_PATH)
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xlrd"])
    df = pd.read_excel(KNOTINFO_XLS_PATH)

cols = list(df.columns)

def pick_col_contains_all(tokens):
    toks = [t.lower() for t in tokens]
    for c in cols:
        cl = c.lower()
        if all(t in cl for t in toks):
            return c
    return None

# knot name column
knot_col = None
for c in cols:
    if c.lower() in {"knot", "name", "knot_name", "knotname", "id"}:
        knot_col = c
        break
if knot_col is None:
    knot_col = cols[0]

# unknotting number column
u_col = None
for pat in (["unknotting", "number"], ["unknotting"], ["unknotting_number"], ["u("]):
    u_col = pick_col_contains_all(pat)
    if u_col:
        break

# Jones vector columns
jones_cols = [c for c in cols if ("jones" in c.lower() and ("vector" in c.lower() or "coeff" in c.lower() or "coefficient" in c.lower()))]
if not jones_cols:
    jones_cols = [c for c in cols if "jones" in c.lower()]
if not jones_cols:
    raise RuntimeError("Could not find any Jones-related column in KnotInfo XLS.")

print("Using knot column:", knot_col)
print("Using unknotting number column:", u_col)
print("Candidate Jones columns:", jones_cols[:8], ("(+%d more)" % (len(jones_cols)-8) if len(jones_cols) > 8 else ""))

_int_pat = re.compile(r"-?\d+")

def parse_vector_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, tuple)):
        try:
            return [int(v) for v in x]
        except Exception:
            return None
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [int(z) for z in v]
        except Exception:
            pass
    nums = _int_pat.findall(s)
    if not nums:
        return None
    return [int(z) for z in nums]

def ensure_minmax_coeffs(v):
    """Require [min,max,coeffs...] with length == abs(max-min)+1."""
    if v is None:
        return None
    v = [int(x) for x in v]
    if len(v) < 3:
        return None
    mn, mx = v[0], v[1]
    coeffs = v[2:]
    span = abs(mx - mn)
    if len(coeffs) != span + 1:
        return None
    return mn, mx, coeffs

def strip_leading_trailing_zeros(coeffs):
    coeffs = list(map(int, coeffs))
    i, j = 0, len(coeffs)
    while i < j and coeffs[i] == 0:
        i += 1
    while j > i and coeffs[j-1] == 0:
        j -= 1
    out = coeffs[i:j]
    return out if out else [0]

def canon_coeff_key(coeffs):
    return tuple(strip_leading_trailing_zeros(coeffs))

def canon_coeff_key_mirror(coeffs):
    return tuple(reversed(strip_leading_trailing_zeros(coeffs)))

def span_abs(mn, mx):
    return abs(int(mx) - int(mn))

# ----------------------------
# Build lookup: (abs-span, coeff-key) -> candidates
# ----------------------------
lookup = defaultdict(list)
n_added = 0

for _, row in df.iterrows():
    knot_name = row.get(knot_col)
    u_val = row.get(u_col) if u_col is not None else None

    for jc in jones_cols:
        raw = parse_vector_cell(row.get(jc))
        parsed = ensure_minmax_coeffs(raw)
        if parsed is None:
            continue
        mn, mx, coeffs = parsed
        sp = span_abs(mn, mx)

        k = canon_coeff_key(coeffs)
        km = canon_coeff_key_mirror(coeffs)

        rec = {
            "knot": str(knot_name),
            "unknotting_number": None if (u_val is None or (isinstance(u_val, float) and pd.isna(u_val))) else u_val,
            "knotinfo_min": mn,
            "knotinfo_max": mx,
            "source_col": jc,
        }
        lookup[(sp, k)].append({**rec, "mirror": False})
        lookup[(sp, km)].append({**rec, "mirror": True})
        n_added += 1

print("Built lookup entries:", n_added)

# ----------------------------
# Load user vectors: REQUIRED format [min,max,c0,...]
# ----------------------------
user_vecs = []
with open(USER_VECTORS_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            v = ast.literal_eval(line)
        except Exception:
            v = parse_vector_cell(line)
        user_vecs.append(v)

print("Loaded user vectors:", len(user_vecs))

# ----------------------------
# Match
# ----------------------------
matches = []
unmatched = 0
bad_format = 0

for i, vraw in enumerate(user_vecs, start=1):
    v = parse_vector_cell(vraw) if not isinstance(vraw, (list, tuple)) else list(vraw)
    parsed = ensure_minmax_coeffs(v)

    if parsed is None:
        bad_format += 1
        matches.append({"index": i, "status": "bad_format_expected_[min,max,coeffs...]", "input": vraw, "candidates": []})
        continue

    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)

    key = (sp, canon_coeff_key(coeffs))
    cand = lookup.get(key, [])

    if not cand:
        unmatched += 1
        matches.append({
            "index": i,
            "status": "no_match",
            "input_min": mn,
            "input_max": mx,
            "input_span_abs": sp,
            "input_coeffs": coeffs,
            "candidates": [],
        })
    else:
        seen = set()
        out = []
        for c in cand:
            tup = (c["knot"], str(c["unknotting_number"]), c["mirror"])
            if tup in seen:
                continue
            seen.add(tup)
            out.append({
                "knot": c["knot"],
                "unknotting_number": c["unknotting_number"],
                "mirror": c["mirror"],
                "source_col": c["source_col"],
                "knotinfo_min": c["knotinfo_min"],
                "knotinfo_max": c["knotinfo_max"],
            })
        matches.append({
            "index": i,
            "status": "matched",
            "input_min": mn,
            "input_max": mx,
            "input_span_abs": sp,
            "input_coeffs": coeffs,
            "candidates": out,
        })

print(f"Matched {len(user_vecs)-unmatched-bad_format}/{len(user_vecs)} vectors; unmatched={unmatched}, bad_format={bad_format}")

# ----------------------------
# Extract unknotting numbers "whenever applicable"
# ----------------------------
summary = []
for m in matches:
    if m["status"] != "matched":
        summary.append({"index": m["index"], "unknotting_numbers": [], "note": m["status"]})
        continue
    uvals = []
    for c in m["candidates"]:
        u = c["unknotting_number"]
        if u is None or (isinstance(u, float) and pd.isna(u)):
            continue
        uvals.append(u)
    # de-dup
    seen = set()
    uuniq = []
    for u in uvals:
        su = str(u)
        if su in seen:
            continue
        seen.add(su)
        uuniq.append(u)
    summary.append({"index": m["index"], "unknotting_numbers": uuniq, "n_candidates": len(m["candidates"])})

print("\n=== Unknotting number candidates (per input vector) ===")
for s in summary:
    print(f"#{s['index']:>3}: u candidates = {s.get('unknotting_numbers')}  (note={s.get('note','')})")

# ----------------------------
# Save outputs to OUT_DIR (deterministic)
# ----------------------------
out_matches_jsonl = os.path.join(OUT_DIR, "knotinfo_match_results.jsonl")
out_summary_jsonl = os.path.join(OUT_DIR, "knotinfo_unknotting_candidates.jsonl")

with open(out_matches_jsonl, "w") as f:
    for m in matches:
        f.write(json.dumps(m) + "\n")
print("\nWrote detailed match results to:", out_matches_jsonl)

with open(out_summary_jsonl, "w") as f:
    for s in summary:
        f.write(json.dumps(s) + "\n")
print("Wrote unknotting-number summary to:", out_summary_jsonl)

KNOTINFO_XLS_PATH = /content/drive/MyDrive/crossing-reduction-unknotting/knotinfo_data_complete.xls
USER_VECTORS_PATH = /content/drive/MyDrive/crossing-reduction-unknotting/min_pd_results_flips1_jones_vec_knotinfo.txt
Using knot column: name
Using unknotting number column: unknotting_number
Candidate Jones columns: ['jones_polynomial_vector', 'jones_polynomial_vector_anon'] 


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Built lookup entries: 2977
Loaded user vectors: 2290
Matched 506/2290 vectors; unmatched=1771, bad_format=13

=== Unknotting number candidates (per input vector) ===
#  1: u candidates = [3]  (note=)
#  2: u candidates = [3]  (note=)
#  3: u candidates = [3]  (note=)
#  4: u candidates = [3]  (note=)
#  5: u candidates = []  (note=no_match)
#  6: u candidates = []  (note=no_match)
#  7: u candidates = []  (note=no_match)
#  8: u candidates = []  (note=no_match)
#  9: u candidates = []  (note=no_match)
# 10: u candidates = []  (note=no_match)
# 11: u candidates = []  (note=no_match)
# 12: u candidates = []  (note=no_match)
# 13: u candidates = []  (note=no_match)
# 14: u candidates = []  (note=no_match)
# 15: u candidates = []  (note=no_match)
# 16: u candidates = []  (note=no_match)
# 17: u candidates = []  (note=no_match)
# 18: u candidates = []  (note=no_match)
# 19: u candidates = []  (note=no_match)
# 20: u candidates = []  (note=no_match)
# 21: u candidates = []  (note=no_match)
#

In [ ]:
import glob, ast, re

# --- your target KnotInfo-style vector ---
target_vec = [-7, 2, -2, 4, -6, 9, -9, 9, -8, 6, -3, 1]
target_coeffs = target_vec[2:]
target_coeffs_mirror = list(reversed(target_coeffs))

def extract_first_int_list(line: str):
    """
    Extract the first [...] from a line and parse it as a Python list of ints.
    Works even if the line contains other text/columns.
    """
    m = re.search(r"\[[^\]]+\]", line)
    if not m:
        return None
    s = m.group(0)
    try:
        v = ast.literal_eval(s)
    except Exception:
        return None
    if not isinstance(v, list) or not all(isinstance(x, int) for x in v):
        return None
    return v

# pick newest vector file (same naming pattern your notebook uses)
paths = sorted(glob.glob(f"{OUT_DIR}/min_pd_results_flips*_jones_vec_knotinfo.txt"))
if not paths:
    raise FileNotFoundError(f"No min_pd_results_flips*_jones_vec_knotinfo.txt found in OUT_DIR={OUT_DIR}")
path = paths[-1]
print("Searching in:", path)

hits = []
with open(path, "r", encoding="utf-8") as f:
    for lineno, line in enumerate(f, 1):
        v = extract_first_int_list(line.strip())
        if v is None:
            continue

        # Accept either:
        #  - full KnotInfo vector: [min,max,c...,c]
        #  - coefficients-only vector: [c...,c]
        if len(v) >= 3 and isinstance(v[0], int) and isinstance(v[1], int):
            coeffs = v[2:]
        else:
            coeffs = v

        if coeffs == target_coeffs:
            hits.append((lineno, "direct (up to q-shift)"))
        elif coeffs == target_coeffs_mirror:
            hits.append((lineno, "mirror (up to q-shift)"))

if hits:
    print(f"FOUND {len(hits)} match(es):")
    for lineno, kind in hits[:50]:
        print(f"  line {lineno}: {kind}")
else:
    print("No match (even up to mirror + q-shift).")

Searching in: /content/drive/MyDrive/crossing-reduction-unknotting/min_pd_results_flips1_jones_vec_knotinfo.txt
FOUND 2 match(es):
  line 269: mirror (up to q-shift)
  line 1057: mirror (up to q-shift)
